# Project 10 — Build and Evaluate the Transformer RAG Assistant

Run this notebook **one cell at a time, from top to bottom**.

This version uses:

- your local portfolio repositories as the RAG corpus,
- the **NVIDIA GeForce RTX 5090** through CUDA,
- MiniLM Transformer embeddings,
- optional E5-small and cross-encoder retrieval comparisons,
- FLAN-T5-base answer generation,
- NLI-based groundedness and citation evaluation,
- JSON, CSV, and PNG artifact export for GitHub and Vercel.

> Only public portfolio documentation is collected. Do not include Veralto/Hach files, GCS data, emails, private reports, credentials, or proprietary documents.


## Cell 1 — Configuration and project paths


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from collections import defaultdict

import numpy as np
import pandas as pd


def locate_project_root() -> Path:
    """Locate 10-ai-portfolio-rag-assistant even when Jupyter starts in /notebooks."""
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name == "10-ai-portfolio-rag-assistant":
            return candidate

    direct_candidate = current / "10-ai-portfolio-rag-assistant"
    if direct_candidate.exists():
        return direct_candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the 10-ai-portfolio-rag-assistant directory. "
        f"Current working directory: {current}"
    )


PROJECT_ROOT = locate_project_root()

# Expected layout:
# GIT Projects/
# ├── ann-deep-learning-projects/
# ├── simple-rnn-projects/
# ├── lstm-projects/
# ├── bi-directional-lstm-projects/
# ├── cnn-projects/
# └── transformer-projects/
#     └── 10-ai-portfolio-rag-assistant/
PORTFOLIO_ROOT = PROJECT_ROOT.parent.parent

RAW_DOCS_ROOT = PROJECT_ROOT / "data" / "raw_portfolio_docs"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PUBLIC_DATA_ROOT = PROJECT_ROOT / "public" / "data"
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"

LOCAL_REPOSITORIES = {
    "ANN": PORTFOLIO_ROOT / "ann-deep-learning-projects",
    "Simple RNN": PORTFOLIO_ROOT / "simple-rnn-projects",
    "LSTM": PORTFOLIO_ROOT / "lstm-projects",
    "BiLSTM": PORTFOLIO_ROOT / "bi-directional-lstm-projects",
    "CNN": PORTFOLIO_ROOT / "cnn-projects",
    "Transformer": PROJECT_ROOT.parent,
}

# Main experiment configuration
CLEAN_LOCAL_CORPUS = False
EMBEDDING_PROVIDER = "minilm"       # "minilm" or "e5"
GENERATOR_MODE = "flan-t5-base"     # "flan-t5-base" or "extractive"
INCLUDE_E5_COMPARISON = True
INCLUDE_CROSS_ENCODER_RERANKER = True

CHUNK_SIZE_WORDS = 220
CHUNK_OVERLAP_WORDS = 50
TOP_K = 5
BATCH_SIZE = 64
MIN_RETRIEVAL_SCORE = 0.20

PYTHON = sys.executable
NPM = "npm.cmd" if os.name == "nt" else "npm"

for folder in [RAW_DOCS_ROOT, PROCESSED_ROOT, PUBLIC_DATA_ROOT, OUTPUTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root       :", PROJECT_ROOT)
print("Portfolio root     :", PORTFOLIO_ROOT)
print("Python executable  :", PYTHON)
print("Python version     :", sys.version.split()[0])
print("Raw corpus folder  :", RAW_DOCS_ROOT)


Project root       : C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant
Portfolio root     : C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects
Python executable  : C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe
Python version     : 3.12.10
Raw corpus folder  : C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\raw_portfolio_docs


## Cell 2 — RTX 5090 and CUDA verification


In [2]:
import torch

print("PyTorch version     :", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available      :", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "CUDA is not available. Stop here because this notebook is intended to use the RTX GPU."
)

DEVICE = "cuda:0"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("Selected device     :", DEVICE)
print("GPU                  :", GPU_NAME)
print("GPU memory (GB)      :", round(GPU_MEMORY_GB, 2))
print("Compute capability   :", torch.cuda.get_device_capability(0))
print("Supported arch list  :", torch.cuda.get_arch_list())

assert "RTX 5090" in GPU_NAME, f"Unexpected GPU detected: {GPU_NAME}"

torch.set_float32_matmul_precision("high")

# Real GPU calculation
x = torch.randn((4096, 4096), device=DEVICE)
y = x @ x
torch.cuda.synchronize()

print("Test tensor device   :", y.device)
print("Allocated GPU memory :", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

del x, y
torch.cuda.empty_cache()

print("RTX 5090 verification passed.")


PyTorch version     : 2.13.0+cu130
PyTorch CUDA runtime: 13.0
CUDA available      : True
Selected device     : cuda:0
GPU                  : NVIDIA GeForce RTX 5090
GPU memory (GB)      : 31.84
Compute capability   : (12, 0)
Supported arch list  : ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Test tensor device   : cuda:0
Allocated GPU memory : 0.156 GB
RTX 5090 verification passed.


## Cell 3 — Reproducible command helper


In [3]:
def run_command(args: list[str]) -> None:
    """Run a project command with the active Project 10 Python environment."""
    printable = " ".join(f'"{item}"' if " " in str(item) else str(item) for item in args)
    print(f"\n$ {printable}\n")
    subprocess.run(
        [str(item) for item in args],
        cwd=PROJECT_ROOT,
        check=True,
    )


print("Command helper ready.")


Command helper ready.


## Cell 4 — Collect portfolio documentation from local repositories

This replaces the GitHub API collector. It copies only recruiter-relevant Markdown documentation from your local repositories.

The following directories are excluded:

- `.git`
- `.venv`
- `node_modules`
- datasets and generated data
- output folders
- model checkpoints
- build folders


In [4]:
EXCLUDED_DIRECTORIES = {
    ".git",
    ".github",
    ".venv",
    "venv",
    "env",
    "node_modules",
    ".next",
    "dist",
    "build",
    "__pycache__",
    ".ipynb_checkpoints",
    "data",
    "datasets",
    "outputs",
    "public",
    "models",
    "checkpoints",
    "artifacts",
    "cache",
}

SPECIAL_DOCUMENT_NAMES = {
    "MODEL_CARD.MD",
    "DATASET_CARD.MD",
    "PROJECT_ROADMAP.MD",
    "RESULTS.MD",
    "EVALUATION.MD",
    "LIMITATIONS.MD",
    "MANUAL_ERROR_ANALYSIS.MD",
}


def iter_supported_markdown(repository_path: Path):
    """Yield supported Markdown files while pruning large/generated directories."""
    for current_root, directory_names, file_names in os.walk(repository_path):
        directory_names[:] = [
            name
            for name in directory_names
            if name.lower() not in EXCLUDED_DIRECTORIES
        ]

        current_path = Path(current_root)

        for file_name in file_names:
            file_name_upper = file_name.upper()

            if not file_name.lower().endswith(".md"):
                continue

            is_readme = file_name_upper.startswith("README")
            is_special = file_name_upper in SPECIAL_DOCUMENT_NAMES

            if is_readme or is_special:
                yield current_path / file_name


missing_repositories = {
    category: path
    for category, path in LOCAL_REPOSITORIES.items()
    if not path.exists()
}

if missing_repositories:
    print("Missing local repositories:")
    for category, path in missing_repositories.items():
        print(f" - {category}: {path}")

    raise FileNotFoundError(
        "One or more local repositories were not found. "
        "Correct LOCAL_REPOSITORIES in Cell 1 before continuing."
    )

if CLEAN_LOCAL_CORPUS and RAW_DOCS_ROOT.exists():
    print("Cleaning existing local corpus:", RAW_DOCS_ROOT)
    shutil.rmtree(RAW_DOCS_ROOT)
    RAW_DOCS_ROOT.mkdir(parents=True, exist_ok=True)

copied_by_category = defaultdict(int)
copied_files: list[Path] = []

for category, repository_path in LOCAL_REPOSITORIES.items():
    repository_path = repository_path.resolve()
    print(f"\nScanning {category}: {repository_path}")

    for source_file in iter_supported_markdown(repository_path):
        relative_path = source_file.relative_to(repository_path)

        destination_file = (
            RAW_DOCS_ROOT
            / category
            / repository_path.name
            / relative_path
        )

        destination_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_file, destination_file)

        copied_by_category[category] += 1
        copied_files.append(destination_file)

print("\nLocal corpus collection summary")
print("=" * 68)

for category in LOCAL_REPOSITORIES:
    print(f"{category:15s}: {copied_by_category[category]:4d} documents")

print("=" * 68)
print("Total copied documents:", len(copied_files))

assert copied_files, "No Markdown documents were copied."



Scanning ANN: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\ann-deep-learning-projects

Scanning Simple RNN: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\simple-rnn-projects

Scanning LSTM: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\lstm-projects

Scanning BiLSTM: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\bi-directional-lstm-projects

Scanning CNN: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\cnn-projects

Scanning Transformer: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects

Local corpus collection summary
ANN            :   35 documents
Simple RNN     :   21 documents
LSTM           :   36 documents
BiLSTM         :   23 documents
CNN            :   38 documents
Transformer    :   69 documents
Total copied documents: 222


## Cell 5 — Verify corpus coverage before preprocessing


In [5]:
raw_documents = sorted(RAW_DOCS_ROOT.rglob("*.md"))

category_rows = []
for category in LOCAL_REPOSITORIES:
    category_path = RAW_DOCS_ROOT / category
    category_rows.append(
        {
            "category": category,
            "document_count": len(list(category_path.rglob("*.md"))) if category_path.exists() else 0,
        }
    )

category_df = pd.DataFrame(category_rows)
display(category_df)

print("Total raw documents:", len(raw_documents))
print("Corpus location    :", RAW_DOCS_ROOT)

required_categories = set(LOCAL_REPOSITORIES)
available_categories = {
    row["category"]
    for row in category_rows
    if row["document_count"] > 0
}
missing_categories = required_categories - available_categories

assert raw_documents, "The raw portfolio corpus is empty."
assert not missing_categories, f"Missing corpus categories: {sorted(missing_categories)}"

sample_rows = []
for path in raw_documents[:20]:
    sample_rows.append(
        {
            "relative_path": path.relative_to(RAW_DOCS_ROOT).as_posix(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
    )

display(pd.DataFrame(sample_rows))
print("All six portfolio categories are present.")


,category,document_count
0,ANN,50
1,Simple RNN,27
2,LSTM,36
3,BiLSTM,23
4,CNN,38
5,Transformer,69


Total raw documents: 243
Corpus location    : C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\raw_portfolio_docs


,relative_path,size_kb
0,ANN/ann-deep-learning-projects/01-churn-classi...,0.30
1,ANN/ann-deep-learning-projects/01-churn-classi...,1.22
2,ANN/ann-deep-learning-projects/01-churn-classi...,1.90
3,ANN/ann-deep-learning-projects/01-churn-classi...,9.43
4,ANN/ann-deep-learning-projects/02-credit-card-...,0.29
5,ANN/ann-deep-learning-projects/02-credit-card-...,2.10
6,ANN/ann-deep-learning-projects/02-credit-card-...,1.19
7,ANN/ann-deep-learning-projects/02-credit-card-...,0.75
8,ANN/ann-deep-learning-projects/02-credit-card-...,15.17
9,ANN/ann-deep-learning-projects/02-credit-card-...,6.41


All six portfolio categories are present.


## Cell 6 — Preprocess and section-chunk the Markdown corpus


In [6]:
run_command([
    PYTHON,
    "scripts/prepare_corpus.py",
    "--input", str(RAW_DOCS_ROOT),
    "--output", str(PROCESSED_ROOT),
    "--chunk-size", str(CHUNK_SIZE_WORDS),
    "--overlap", str(CHUNK_OVERLAP_WORDS),
])

stats_path = OUTPUTS_ROOT / "corpus_statistics.json"
metadata_path = PROCESSED_ROOT / "metadata.json"
corpus_path = PROCESSED_ROOT / "portfolio_corpus.json"
chunks_path = PROCESSED_ROOT / "document_chunks.json"

stats = json.loads(stats_path.read_text(encoding="utf-8"))
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
portfolio_corpus = json.loads(corpus_path.read_text(encoding="utf-8"))
document_chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

print(json.dumps(stats, indent=2))
print("Categories   :", metadata.get("categories"))
print("Repositories :", metadata.get("sourceRepositories"))

assert stats["document_count"] > 0, "No documents were processed."
assert stats["chunk_count"] > 0, "No document chunks were created."
assert len(portfolio_corpus) == stats["document_count"]
assert len(document_chunks) == stats["chunk_count"]



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/prepare_corpus.py --input "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\raw_portfolio_docs" --output "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed" --chunk-size 220 --overlap 50

{
  "document_count": 220,
  "chunk_count": 3157,
  "category_document_counts": {
    "ANN": 40,
    "BiLSTM": 23,
    "CNN": 37,
    "LSTM": 34,
    "Simple RNN": 23,
    "Transformer": 63
  },
  "deployment_document_counts": {
    "Not specified": 44,
    "Streamlit": 84,
    "Hugging Face": 49,
    "Vercel": 19,
    "Gradio": 4,
    "GitHub Pages": 19,
    "TensorFlow.js": 1
  },
  "mean_chunk_words": 55.72,
  "min_chunk_words": 1,
  "max_chunk_words": 220
}
Categories   : ['ANN', 'BiLSTM

## Cell 7 — Inspect processed documents and chunks


In [7]:
document_preview = pd.DataFrame(
    [
        {
            "project_id": item["project_id"],
            "repository": item["repository"],
            "category": item["category"],
            "deployment": item["deployment"],
            "source_file": item["source_file"],
        }
        for item in portfolio_corpus[:25]
    ]
)

chunk_preview = pd.DataFrame(
    [
        {
            "chunk_id": item["id"],
            "project_id": item["projectId"],
            "category": item["category"],
            "section": item["section"],
            "word_count": len(item["text"].split()),
            "source_file": item["sourceFile"],
        }
        for item in document_chunks[:25]
    ]
)

print("Processed document preview")
display(document_preview)

print("Chunk preview")
display(chunk_preview)

category_chunk_counts = (
    pd.DataFrame(document_chunks)
    .groupby("category", dropna=False)
    .size()
    .reset_index(name="chunk_count")
    .sort_values("chunk_count", ascending=False)
)
display(category_chunk_counts)


Processed document preview


,project_id,repository,category,deployment,source_file
0,.pytest_cache,ann-deep-learning-projects,ANN,Not specified,README.md
1,data,ann-deep-learning-projects,ANN,Not specified,README.md
2,01-churn-classification,ann-deep-learning-projects,ANN,Streamlit,README.md
3,data,ann-deep-learning-projects,ANN,Streamlit,README_data.md
4,models,ann-deep-learning-projects,ANN,Streamlit,README_models.md
5,outputs,ann-deep-learning-projects,ANN,Not specified,README_outputs.md
6,02-credit-card-fraud-detection,ann-deep-learning-projects,ANN,Streamlit,README.md
7,02-credit-card-fraud-detection,ann-deep-learning-projects,ANN,Streamlit,README_HOSTING.md
8,data,ann-deep-learning-projects,ANN,Streamlit,README_data.md
9,03-credit-risk-probability-scoring,ann-deep-learning-projects,ANN,Streamlit,MODEL_CARD.md


Chunk preview


,chunk_id,project_id,category,section,word_count,source_file
0,ann-deep-learning-projects:.pytest_cache:READM...,.pytest_cache,ANN,pytest cache directory #,35,README.md
1,ann-deep-learning-projects:data:README.md:61f9...,data,ANN,Included Files,16,README.md
2,ann-deep-learning-projects:data:README.md:61f9...,data,ANN,Target,9,README.md
3,ann-deep-learning-projects:data:README.md:61f9...,data,ANN,Prediction Features,85,README.md
4,ann-deep-learning-projects:data:README.md:61f9...,data,ANN,Excluded Identifiers,9,README.md
5,ann-deep-learning-projects:data:README.md:61f9...,data,ANN,Before Public Release,58,README.md
6,ann-deep-learning-projects:01-churn-classifica...,01-churn-classification,ANN,Customer Churn Classification using Artificial...,86,README.md
7,ann-deep-learning-projects:01-churn-classifica...,01-churn-classification,ANN,Project Overview,61,README.md
8,ann-deep-learning-projects:01-churn-classifica...,01-churn-classification,ANN,Business Problem,59,README.md
9,ann-deep-learning-projects:01-churn-classifica...,01-churn-classification,ANN,Low-Risk Customer,4,README.md


,category,chunk_count
5,Transformer,741
0,ANN,588
3,LSTM,551
1,BiLSTM,469
2,CNN,419
4,Simple RNN,389


## Cell 8 — Generate real MiniLM Transformer embeddings on the RTX 5090

This cell must produce metadata containing:

- provider: `huggingface-feature-extraction`
- model: `sentence-transformers/all-MiniLM-L6-v2`
- dimension: `384`
- normalized: `true`


In [8]:
run_command([
    PYTHON,
    "scripts/generate_embeddings.py",
    "--provider", EMBEDDING_PROVIDER,
    "--device", DEVICE,
    "--batch-size", str(BATCH_SIZE),
    "--input", str(PROCESSED_ROOT / "document_chunks.json"),
    "--output", str(PROCESSED_ROOT / "embeddings.json"),
])

metadata = json.loads(
    (PROCESSED_ROOT / "metadata.json").read_text(encoding="utf-8")
)
embedding_records = json.loads(
    (PROCESSED_ROOT / "embeddings.json").read_text(encoding="utf-8")
)

print(json.dumps(metadata["embedding"], indent=2))

assert metadata["embedding"]["provider"] == "huggingface-feature-extraction"
assert "hash" not in metadata["embedding"]["model"].lower()
assert len(embedding_records) == len(document_chunks)

embedding_matrix = np.asarray(
    [record["vector"] for record in embedding_records],
    dtype=np.float32,
)

embedding_norms = np.linalg.norm(embedding_matrix, axis=1)

print("Embedding matrix shape :", embedding_matrix.shape)
print("Mean vector norm       :", round(float(embedding_norms.mean()), 6))
print("Minimum vector norm    :", round(float(embedding_norms.min()), 6))
print("Maximum vector norm    :", round(float(embedding_norms.max()), 6))

assert embedding_matrix.shape[1] == metadata["embedding"]["dimension"]
assert np.allclose(embedding_norms, 1.0, atol=1e-3)

print("Real Transformer embeddings generated successfully on", DEVICE)



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/generate_embeddings.py --provider minilm --device cuda:0 --batch-size 64 --input "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\document_chunks.json" --output "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\embeddings.json"

{
  "provider": "huggingface-feature-extraction",
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "dimension": 384,
  "normalized": true,
  "queryPrefix": "",
  "passagePrefix": "",
  "generatedAt": "2026-07-30T16:02:22.096891+00:00",
  "documentEmbeddingsPrecomputed": true
}
Embedding matrix shape : (3157, 384)
Mean vector norm       : 1.0
Minimum vector norm    : 1.0
Maximum vector norm    : 1.0
Real Transformer embeddings generat

## Cell 9 — Export the static vector store for Next.js and Vercel


In [9]:
run_command([
    PYTHON,
    "scripts/export_vector_store.py",
])

required_public_files = [
    "document_chunks.json",
    "embeddings.json",
    "metadata.json",
    "evaluation_questions.json",
]

public_file_rows = []

for file_name in required_public_files:
    path = PUBLIC_DATA_ROOT / file_name

    public_file_rows.append(
        {
            "file": file_name,
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else 0,
        }
    )

public_files_df = pd.DataFrame(public_file_rows)
display(public_files_df)

assert public_files_df["exists"].all(), "One or more Vercel data files are missing."



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/export_vector_store.py



,file,exists,size_mb
0,document_chunks.json,True,5.660
1,embeddings.json,True,22.791
2,metadata.json,True,0.002
3,evaluation_questions.json,True,0.017


## Cell 10 — Benchmark retrieval methods

This compares:

1. TF-IDF keyword retrieval
2. Hash-vector baseline
3. MiniLM dense retrieval
4. E5-small dense retrieval
5. MiniLM plus cross-encoder reranking


In [10]:
benchmark_command = [
    PYTHON,
    "scripts/run_retrieval_benchmark.py",
    "--chunks", str(PROCESSED_ROOT / "document_chunks.json"),
    "--questions", str(PROCESSED_ROOT / "evaluation_questions.json"),
    "--output", str(OUTPUTS_ROOT / "retrieval_benchmark.json"),
    "--device", DEVICE,
    "--k", "1", "3", "5", "10",
    "--candidate-k", "20",
]

if INCLUDE_E5_COMPARISON:
    benchmark_command.append("--include-e5")

if INCLUDE_CROSS_ENCODER_RERANKER:
    benchmark_command.append("--include-reranker")

run_command(benchmark_command)



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/run_retrieval_benchmark.py --chunks "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\document_chunks.json" --questions "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\evaluation_questions.json" --output "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\outputs\retrieval_benchmark.json" --device cuda:0 --k 1 3 5 10 --candidate-k 20 --include-e5 --include-reranker



## Cell 11 — Inspect retrieval metrics


In [11]:
retrieval = json.loads(
    (OUTPUTS_ROOT / "retrieval_benchmark.json").read_text(encoding="utf-8")
)

retrieval_rows = []

for method in retrieval["methods"]:
    for k_name, metric_values in method["summary"].items():
        retrieval_rows.append(
            {
                "method": method["method"],
                "k": k_name,
                **metric_values,
                **method["latency_ms"],
            }
        )

retrieval_df = pd.DataFrame(retrieval_rows)
display(retrieval_df)

k5 = (
    retrieval_df[retrieval_df["k"] == "k=5"]
    .sort_values(
        ["recall", "ndcg", "mrr"],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Top retrieval methods at K=5")
display(k5)

assert not k5.empty, "No K=5 retrieval metrics were generated."


,method,k,hit_rate,precision,recall,mrr,map,ndcg,query_embedding_mean,query_embedding_p95,retrieval_mean,retrieval_p95
0,tfidf-keyword,k=1,0.514286,0.514286,0.450000,0.514286,0.514286,0.514286,0.230,0.347,0.580,0.881
1,tfidf-keyword,k=3,0.685714,0.238095,0.569048,0.595238,0.515873,0.551217,0.230,0.347,0.580,0.881
2,tfidf-keyword,k=5,0.742857,0.177143,0.640476,0.611905,0.537897,0.584254,0.230,0.347,0.580,0.881
3,tfidf-keyword,k=10,0.771429,0.100000,0.692857,0.616667,0.550550,0.605076,0.230,0.347,0.580,0.881
4,hash-vector,k=1,0.285714,0.285714,0.219048,0.285714,0.285714,0.285714,0.020,0.045,0.157,0.222
5,hash-vector,k=3,0.457143,0.161905,0.361905,0.361905,0.295238,0.330457,0.020,0.045,0.157,0.222
6,hash-vector,k=5,0.571429,0.120000,0.453333,0.397619,0.316667,0.369261,0.020,0.045,0.157,0.222
7,hash-vector,k=10,0.657143,0.077143,0.558095,0.409320,0.334762,0.407212,0.020,0.045,0.157,0.222
8,minilm-dense,k=1,0.428571,0.428571,0.407143,0.428571,0.428571,0.428571,4.717,6.395,0.265,0.360
9,minilm-dense,k=3,0.657143,0.228571,0.555714,0.523809,0.482540,0.516819,4.717,6.395,0.265,0.360


Top retrieval methods at K=5


,method,k,hit_rate,precision,recall,mrr,map,ndcg,query_embedding_mean,query_embedding_p95,retrieval_mean,retrieval_p95
0,minilm-plus-cross-encoder,k=5,0.742857,0.171429,0.646190,0.617619,0.557651,0.596122,5.594,10.354,19.079,25.416
1,tfidf-keyword,k=5,0.742857,0.177143,0.640476,0.611905,0.537897,0.584254,0.230,0.347,0.580,0.881
2,e5-small-v2,k=5,0.714286,0.154286,0.623810,0.650000,0.556667,0.594230,6.653,10.607,0.289,0.412
3,minilm-dense,k=5,0.714286,0.148571,0.593810,0.535238,0.481667,0.524242,4.717,6.395,0.265,0.360
4,hash-vector,k=5,0.571429,0.120000,0.453333,0.397619,0.316667,0.369261,0.020,0.045,0.157,0.222


## Cell 12 — Generate answers and evaluate groundedness, citations, refusals, and latency

This stage uses:

- MiniLM retrieval
- FLAN-T5-base generation
- `cross-encoder/nli-deberta-v3-small` for claim-level support evaluation

The first run downloads the Hugging Face models. Keep `nvidia-smi -l 1` open in a second Command Prompt to observe GPU use.


In [13]:
run_command([
    PYTHON,
    "scripts/evaluate_answers.py",
    "--chunks", str(PROCESSED_ROOT / "document_chunks.json"),
    "--questions", str(PROCESSED_ROOT / "evaluation_questions.json"),
    "--output-dir", str(OUTPUTS_ROOT),
    "--generator", GENERATOR_MODE,
    "--device", DEVICE,
    "--top-k", str(TOP_K),
    "--min-retrieval-score", str(MIN_RETRIEVAL_SCORE),
])



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/evaluate_answers.py --chunks "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\document_chunks.json" --questions "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\data\processed\evaluation_questions.json" --output-dir "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\outputs" --generator flan-t5-base --device cuda:0 --top-k 5 --min-retrieval-score 0.2



## Cell 13 — Inspect answer-quality and latency results


In [14]:
groundedness = json.loads(
    (OUTPUTS_ROOT / "answer_groundedness_results.json").read_text(encoding="utf-8")
)
citations = json.loads(
    (OUTPUTS_ROOT / "citation_correctness_results.json").read_text(encoding="utf-8")
)
latency = json.loads(
    (OUTPUTS_ROOT / "response_latency_results.json").read_text(encoding="utf-8")
)

print("Groundedness summary")
print(json.dumps(groundedness["summary"], indent=2))

print("\nCitation summary")
print(json.dumps(citations["summary"], indent=2))

print("\nLatency summary")
print(json.dumps(latency["summary"], indent=2))

answer_metrics_df = pd.DataFrame(
    [
        {
            "metric": "Mean groundedness",
            "value": groundedness["summary"].get("mean_groundedness"),
        },
        {
            "metric": "Citation precision",
            "value": groundedness["summary"].get("mean_citation_precision"),
        },
        {
            "metric": "Citation completeness",
            "value": groundedness["summary"].get("mean_citation_completeness"),
        },
        {
            "metric": "Unsupported claim rate",
            "value": groundedness["summary"].get("mean_unsupported_claim_rate"),
        },
        {
            "metric": "Refusal accuracy",
            "value": groundedness["summary"].get("refusal_accuracy"),
        },
        {
            "metric": "Median latency (ms)",
            "value": latency["summary"].get("median_ms"),
        },
        {
            "metric": "P95 latency (ms)",
            "value": latency["summary"].get("p95_ms"),
        },
    ]
)

display(answer_metrics_df)


Groundedness summary
{
  "answer_count": 40,
  "mean_groundedness": 0.025,
  "mean_citation_precision": 0.0,
  "mean_citation_completeness": 0.275,
  "mean_unsupported_claim_rate": 0.975,
  "refusal_accuracy": 0.875
}

Citation summary
{
  "mean_citation_precision": 0.0,
  "mean_citation_completeness": 0.275,
  "mean_unsupported_claim_rate": 0.975
}

Latency summary
{
  "count": 40,
  "mean_ms": 152.515,
  "median_ms": 113.407,
  "p90_ms": 159.268,
  "p95_ms": 348.339,
  "min_ms": 42.672,
  "max_ms": 1337.153
}


,metric,value
0,Mean groundedness,0.025
1,Citation precision,0.000
2,Citation completeness,0.275
3,Unsupported claim rate,0.975
4,Refusal accuracy,0.875
5,Median latency (ms),113.407
6,P95 latency (ms),348.339


## Cell 14 — Build evaluation summary and charts


In [15]:
run_command([
    PYTHON,
    "scripts/build_evaluation_summary.py",
])

run_command([
    PYTHON,
    "scripts/create_evaluation_charts.py",
])

evaluation_summary_path = PUBLIC_DATA_ROOT / "evaluation_summary.json"
evaluation_summary = json.loads(
    evaluation_summary_path.read_text(encoding="utf-8")
)

print(json.dumps(evaluation_summary, indent=2))

expected_charts = [
    OUTPUTS_ROOT / "retrieval_method_comparison.png",
    OUTPUTS_ROOT / "rag_answer_quality.png",
    OUTPUTS_ROOT / "response_latency_distribution.png",
]

chart_rows = [
    {
        "chart": chart.name,
        "exists": chart.exists(),
        "size_kb": round(chart.stat().st_size / 1024, 2) if chart.exists() else 0,
    }
    for chart in expected_charts
]

display(pd.DataFrame(chart_rows))



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/build_evaluation_summary.py


$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" scripts/create_evaluation_charts.py

{
  "status": "measured",
  "generated_at": "2026-07-30T16:13:13.687151+00:00",
  "corpus": {
    "coverage_status": "complete",
    "document_count": 220,
    "chunk_count": 3157,
    "categories": [
      "ANN",
      "BiLSTM",
      "CNN",
      "LSTM",
      "Simple RNN",
      "Transformer"
    ]
  },
  "models": {
    "embedding": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_provider": "huggingface-feature-extraction",
    "groundedness_evaluator": "cross-encoder/nli-deberta-v3-small"
  },
  "retrieval": {
    "best_method": "minilm-plus-cross-encoder",
    "hit_rate_at_5": 0.742857,
    "precision_at_5

,chart,exists,size_kb
0,retrieval_method_comparison.png,True,62.83
1,rag_answer_quality.png,True,49.03
2,response_latency_distribution.png,True,47.02


## Cell 15 — Apply the portfolio quality gates

These are target thresholds, not fabricated results. A failed gate identifies the part of the system that should be improved.


In [16]:
best_k5 = k5.iloc[0].to_dict() if not k5.empty else {}
ground_summary = groundedness.get("summary", {})
required_categories = {
    "ANN",
    "Simple RNN",
    "LSTM",
    "BiLSTM",
    "CNN",
    "Transformer",
}

quality_gates = {
    "Real Transformer embeddings": (
        metadata["embedding"]["provider"] == "huggingface-feature-extraction"
    ),
    "At least 40 evaluation questions": (
        retrieval.get("question_count", 0)
        + retrieval.get("unsupported_question_count", 0)
        >= 40
    ),
    "Recall@5 at least 0.80": (
        best_k5.get("recall", 0) >= 0.80
    ),
    "nDCG@5 at least 0.75": (
        best_k5.get("ndcg", 0) >= 0.75
    ),
    "Groundedness at least 0.85": (
        ground_summary.get("mean_groundedness", 0) >= 0.85
    ),
    "Citation precision at least 0.85": (
        ground_summary.get("mean_citation_precision", 0) >= 0.85
    ),
    "Citation completeness at least 0.85": (
        ground_summary.get("mean_citation_completeness", 0) >= 0.85
    ),
    "Refusal accuracy at least 0.80": (
        (ground_summary.get("refusal_accuracy") or 0) >= 0.80
    ),
    "All six portfolio categories present": (
        required_categories.issubset(set(metadata.get("categories", [])))
    ),
}

quality_gate_df = pd.DataFrame(
    [
        {
            "quality_gate": gate,
            "passed": passed,
        }
        for gate, passed in quality_gates.items()
    ]
)

display(quality_gate_df)

passed_gate_count = int(quality_gate_df["passed"].sum())
total_gate_count = len(quality_gate_df)

print(f"Passed {passed_gate_count} of {total_gate_count} quality gates.")

if passed_gate_count >= 8:
    print("Portfolio readiness: approximately 9/10 target reached.")
elif passed_gate_count >= 6:
    print("Portfolio readiness: strong, but further tuning is recommended.")
else:
    print("Portfolio readiness: continue improving corpus coverage, retrieval, or generation.")


,quality_gate,passed
0,Real Transformer embeddings,True
1,At least 40 evaluation questions,True
2,Recall@5 at least 0.80,False
3,nDCG@5 at least 0.75,False
4,Groundedness at least 0.85,False
5,Citation precision at least 0.85,False
6,Citation completeness at least 0.85,False
7,Refusal accuracy at least 0.80,True
8,All six portfolio categories present,True


Passed 4 of 9 quality gates.
Portfolio readiness: continue improving corpus coverage, retrieval, or generation.


## Cell 16 — Run Python tests


In [17]:
run_command([
    PYTHON,
    "-m",
    "pytest",
    "-q",
])



$ "C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\.venv\Scripts\python.exe" -m pytest -q



## Cell 17 — Skip local Node.js installation

In [19]:
from __future__ import annotations

import shutil


node_path = shutil.which("node")
npm_path = shutil.which("npm.cmd") or shutil.which("npm")

print("Local Node.js validation mode: SKIPPED")
print("Reason: Node.js will not be installed on this office computer.")
print("Node detected:", node_path or "Not installed")
print("npm detected:", npm_path or "Not installed")

print(
    "\nThe Transformer/RAG pipeline and evaluation artifacts "
    "do not require Node.js."
)

print(
    "Next.js dependency installation, type checking, testing, "
    "and production building will be performed remotely using "
    "GitHub Actions and Vercel."
)

LOCAL_NODE_VALIDATION_SKIPPED = True
REMOTE_NODE_VALIDATION_REQUIRED = True

print("\nCell 17 completed successfully.")

Local Node.js validation mode: SKIPPED
Reason: Node.js will not be installed on this office computer.
Node detected: Not installed
npm detected: Not installed

The Transformer/RAG pipeline and evaluation artifacts do not require Node.js.
Next.js dependency installation, type checking, testing, and production building will be performed remotely using GitHub Actions and Vercel.

Cell 17 completed successfully.


## Cell 18 — Python-only Vercel deployment preflight

In [20]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


project_root = PROJECT_ROOT.resolve()
public_data_root = project_root / "public" / "data"
processed_root = project_root / "data" / "processed"
outputs_root = project_root / "outputs"

outputs_root.mkdir(parents=True, exist_ok=True)


required_application_files = [
    project_root / "package.json",
    project_root / "next.config.js",
    project_root / "vercel.json",
    project_root / "tsconfig.json",
    project_root / "app" / "page.tsx",
    project_root / "app" / "layout.tsx",
    project_root / "app" / "globals.css",
    project_root / "app" / "api" / "chat" / "route.ts",
    project_root / "app" / "api" / "retrieve" / "route.ts",
    project_root / "app" / "api" / "health" / "route.ts",
]

required_processed_files = [
    processed_root / "portfolio_corpus.json",
    processed_root / "document_chunks.json",
    processed_root / "embeddings.json",
    processed_root / "metadata.json",
    processed_root / "evaluation_questions.json",
]

required_public_files = [
    public_data_root / "document_chunks.json",
    public_data_root / "embeddings.json",
    public_data_root / "metadata.json",
    public_data_root / "evaluation_questions.json",
]

required_evaluation_files = [
    outputs_root / "retrieval_recall_at_k.json",
    outputs_root / "answer_groundedness_results.json",
    outputs_root / "citation_correctness_results.json",
    outputs_root / "response_latency_results.json",
    outputs_root / "rag_answer_examples.csv",
]


all_required_files = (
    required_application_files
    + required_processed_files
    + required_public_files
    + required_evaluation_files
)

missing_files = [
    str(path.relative_to(project_root))
    for path in all_required_files
    if not path.exists()
]


def load_json(path: Path) -> Any:
    """Load and validate a JSON file."""

    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    except json.JSONDecodeError as exc:
        raise AssertionError(
            f"Invalid JSON file: {path}\n{exc}"
        ) from exc


def extract_records(
    payload: Any,
    possible_keys: tuple[str, ...],
) -> list[Any]:
    """
    Extract a record list from either a direct JSON array
    or a dictionary containing one of the expected keys.
    """

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in possible_keys:
            value = payload.get(key)

            if isinstance(value, list):
                return value

    return []


def extract_vector(record: Any) -> list[float]:
    """Extract an embedding vector from supported record structures."""

    if isinstance(record, list):
        return record

    if isinstance(record, dict):
        for key in (
            "embedding",
            "vector",
            "values",
        ):
            value = record.get(key)

            if isinstance(value, list):
                return value

    return []


if missing_files:
    print("Missing required files:")

    for missing_file in missing_files:
        print(" -", missing_file)

    raise AssertionError(
        f"{len(missing_files)} required deployment files are missing."
    )


print("All required application and artifact files are present.")


# ------------------------------------------------------------------
# Validate package.json without running npm
# ------------------------------------------------------------------

package_json_path = project_root / "package.json"
package_json = load_json(package_json_path)

scripts = package_json.get("scripts", {})
dependencies = {
    **package_json.get("dependencies", {}),
    **package_json.get("devDependencies", {}),
}

required_scripts = {
    "dev",
    "build",
}

missing_scripts = sorted(
    required_scripts - set(scripts)
)

if missing_scripts:
    raise AssertionError(
        "package.json is missing scripts: "
        + ", ".join(missing_scripts)
    )


required_packages = {
    "next",
    "react",
    "react-dom",
    "typescript",
}

missing_packages = sorted(
    required_packages - set(dependencies)
)

if missing_packages:
    raise AssertionError(
        "package.json is missing packages: "
        + ", ".join(missing_packages)
    )


print("package.json structure passed.")
print("Build script:", scripts.get("build"))
print("Next.js version:", dependencies.get("next"))
print("React version:", dependencies.get("react"))


# ------------------------------------------------------------------
# Validate static Vercel data files
# ------------------------------------------------------------------

chunks_payload = load_json(
    public_data_root / "document_chunks.json"
)

embeddings_payload = load_json(
    public_data_root / "embeddings.json"
)

metadata_payload = load_json(
    public_data_root / "metadata.json"
)

questions_payload = load_json(
    public_data_root / "evaluation_questions.json"
)


chunk_records = extract_records(
    chunks_payload,
    (
        "chunks",
        "documents",
        "records",
        "data",
    ),
)

embedding_records = extract_records(
    embeddings_payload,
    (
        "embeddings",
        "vectors",
        "records",
        "data",
    ),
)

evaluation_questions = extract_records(
    questions_payload,
    (
        "questions",
        "evaluation_questions",
        "records",
        "data",
    ),
)


assert chunk_records, (
    "public/data/document_chunks.json contains no chunks."
)

assert embedding_records, (
    "public/data/embeddings.json contains no embeddings."
)

assert evaluation_questions, (
    "public/data/evaluation_questions.json "
    "contains no evaluation questions."
)

assert isinstance(metadata_payload, dict), (
    "public/data/metadata.json must contain a JSON object."
)


chunk_count = len(chunk_records)
embedding_count = len(embedding_records)
question_count = len(evaluation_questions)

assert chunk_count == embedding_count, (
    "Chunk and embedding counts do not match: "
    f"{chunk_count} chunks versus "
    f"{embedding_count} embeddings."
)


first_vector = extract_vector(
    embedding_records[0]
)

assert first_vector, (
    "The first embedding record does not contain a vector."
)

embedding_dimension = len(first_vector)

assert embedding_dimension > 0, (
    "Embedding dimension must be greater than zero."
)


inconsistent_dimensions = []

for index, record in enumerate(
    embedding_records,
    start=1,
):
    vector = extract_vector(record)

    if len(vector) != embedding_dimension:
        inconsistent_dimensions.append(
            {
                "record": index,
                "dimension": len(vector),
            }
        )


assert not inconsistent_dimensions, (
    "Embedding vectors have inconsistent dimensions: "
    f"{inconsistent_dimensions[:10]}"
)


print("\nStatic vector-store validation passed.")
print("Document chunks:", chunk_count)
print("Embeddings:", embedding_count)
print("Embedding dimension:", embedding_dimension)
print("Evaluation questions:", question_count)


# ------------------------------------------------------------------
# Validate API route files contain expected exports
# ------------------------------------------------------------------

route_files = {
    "chat": project_root
    / "app"
    / "api"
    / "chat"
    / "route.ts",

    "retrieve": project_root
    / "app"
    / "api"
    / "retrieve"
    / "route.ts",

    "health": project_root
    / "app"
    / "api"
    / "health"
    / "route.ts",
}


route_checks: dict[str, bool] = {}

for route_name, route_path in route_files.items():
    route_text = route_path.read_text(
        encoding="utf-8",
    )

    has_route_export = (
        "export async function" in route_text
        or "export function" in route_text
    )

    route_checks[route_name] = has_route_export

    assert has_route_export, (
        f"{route_name} route does not contain "
        "an exported route handler."
    )


print("\nAPI route structural validation passed.")

for route_name, passed in route_checks.items():
    print(
        f"{route_name:10s}:",
        "PASS" if passed else "FAIL",
    )


# ------------------------------------------------------------------
# Validate evaluation outputs
# ------------------------------------------------------------------

evaluation_json_files = [
    outputs_root / "retrieval_recall_at_k.json",
    outputs_root / "answer_groundedness_results.json",
    outputs_root / "citation_correctness_results.json",
    outputs_root / "response_latency_results.json",
]

for evaluation_file in evaluation_json_files:
    load_json(evaluation_file)

print("\nEvaluation JSON validation passed.")


# ------------------------------------------------------------------
# Write a deployment-preflight report
# ------------------------------------------------------------------

preflight_report = {
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "status": "python_preflight_passed",

    "local_node_validation": {
        "performed": False,
        "reason": (
            "Node.js is intentionally not installed "
            "on the office computer."
        ),
    },

    "remote_validation_required": {
        "github_actions": True,
        "vercel_build": True,
    },

    "application_files_present": len(
        required_application_files
    ),

    "processed_files_present": len(
        required_processed_files
    ),

    "public_files_present": len(
        required_public_files
    ),

    "evaluation_files_present": len(
        required_evaluation_files
    ),

    "vector_store": {
        "chunk_count": chunk_count,
        "embedding_count": embedding_count,
        "embedding_dimension": embedding_dimension,
        "evaluation_question_count": question_count,
    },

    "api_routes": route_checks,

    "package_json": {
        "build_script": scripts.get("build"),
        "next_version": dependencies.get("next"),
        "react_version": dependencies.get("react"),
    },
}


preflight_output_path = (
    outputs_root
    / "vercel_deployment_preflight.json"
)

with preflight_output_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preflight_report,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\nDeployment preflight report saved:")
print(preflight_output_path)

print(
    "\nPython-only deployment preflight passed."
)

print(
    "The remaining Next.js installation, "
    "type-check, test, and production-build steps "
    "must run remotely in GitHub Actions or Vercel."
)

All required application and artifact files are present.
package.json structure passed.
Build script: next build
Next.js version: 16.2.12
React version: 19.2.8

Static vector-store validation passed.
Document chunks: 3157
Embeddings: 3157
Embedding dimension: 384
Evaluation questions: 40

API route structural validation passed.
chat      : PASS
retrieve  : PASS
health    : PASS

Evaluation JSON validation passed.

Deployment preflight report saved:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\10-ai-portfolio-rag-assistant\outputs\vercel_deployment_preflight.json

Python-only deployment preflight passed.
The remaining Next.js installation, type-check, test, and production-build steps must run remotely in GitHub Actions or Vercel.


## Cell 19 — Final artifact inventory


In [21]:
important_artifacts = [
    PROCESSED_ROOT / "portfolio_corpus.json",
    PROCESSED_ROOT / "document_chunks.json",
    PROCESSED_ROOT / "embeddings.json",
    PROCESSED_ROOT / "metadata.json",
    PROCESSED_ROOT / "evaluation_questions.json",
    PUBLIC_DATA_ROOT / "document_chunks.json",
    PUBLIC_DATA_ROOT / "embeddings.json",
    PUBLIC_DATA_ROOT / "metadata.json",
    PUBLIC_DATA_ROOT / "evaluation_questions.json",
    PUBLIC_DATA_ROOT / "evaluation_summary.json",
    OUTPUTS_ROOT / "corpus_statistics.json",
    OUTPUTS_ROOT / "retrieval_benchmark.json",
    OUTPUTS_ROOT / "answer_groundedness_results.json",
    OUTPUTS_ROOT / "citation_correctness_results.json",
    OUTPUTS_ROOT / "response_latency_results.json",
    OUTPUTS_ROOT / "rag_answer_examples.csv",
    OUTPUTS_ROOT / "retrieval_method_comparison.png",
    OUTPUTS_ROOT / "rag_answer_quality.png",
    OUTPUTS_ROOT / "response_latency_distribution.png",
]

artifact_rows = []

for artifact in important_artifacts:
    artifact_rows.append(
        {
            "artifact": artifact.relative_to(PROJECT_ROOT).as_posix(),
            "exists": artifact.exists(),
            "size_mb": round(artifact.stat().st_size / 1024**2, 3) if artifact.exists() else 0,
        }
    )

artifact_df = pd.DataFrame(artifact_rows)
display(artifact_df)

missing_artifacts = artifact_df.loc[~artifact_df["exists"], "artifact"].tolist()

if missing_artifacts:
    print("Missing artifacts:")
    for artifact in missing_artifacts:
        print(" -", artifact)
else:
    print("All required Project 10 artifacts were generated successfully.")

print("\nFinal embedding model:", metadata["embedding"]["model"])
print("Final embedding provider:", metadata["embedding"]["provider"])
print("Final device used:", DEVICE)
print("Final GPU:", GPU_NAME)


,artifact,exists,size_mb
0,data/processed/portfolio_corpus.json,True,1.680
1,data/processed/document_chunks.json,True,5.660
2,data/processed/embeddings.json,True,22.791
3,data/processed/metadata.json,True,0.002
4,data/processed/evaluation_questions.json,True,0.017
5,public/data/document_chunks.json,True,5.660
6,public/data/embeddings.json,True,22.791
7,public/data/metadata.json,True,0.002
8,public/data/evaluation_questions.json,True,0.017
9,public/data/evaluation_summary.json,True,0.002


All required Project 10 artifacts were generated successfully.

Final embedding model: sentence-transformers/all-MiniLM-L6-v2
Final embedding provider: huggingface-feature-extraction
Final device used: cuda:0
Final GPU: NVIDIA GeForce RTX 5090


## Cell 20 — After all cells pass

1. Review the generated metrics; do not publish placeholder or invented values.
2. Confirm `public/data/metadata.json` names MiniLM or E5, not `local-hash-v1`.
3. Save the notebook.
4. Commit the generated JSON, CSV, and PNG artifacts to GitHub.
5. Deploy `10-ai-portfolio-rag-assistant` as the Vercel Root Directory.
6. Add the Vercel URL to `README.md`.
7. Run the deployed latency benchmark and save its measured output.
